# 第 2 课 · 查询参数与 URL

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：把一个完整网址拆开看明白，并学会“给服务器附加筛选条件”。
> 只引入两个新概念：**URL 的组成**、**查询参数（query string）**。

In [1]:
# 每个 notebook 开头都要重新 import 这一课要用的库
import requests                                   # HTTP 请求库（每本 notebook 都要重新 import）
from urllib.parse import urlparse, urlencode     # Python 标准库：拆网址、拼查询串，完全离线可用
print('requests 版本：', requests.__version__)

requests 版本： 2.34.2


## 2.1 先把网址拆成四块（生活化类比：快递地址）

你寄快递时会写：**省市 + 街道 + 门牌号 + 备注**。网址也一样，由四块拼成。看这个例子：

```
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
\____/   \_____________________/   \____________________________________/
 协议          主机 host                     路径 path
scheme

https://httpbin.org/get?q=python&page=2
\____/   \____________/   \__/ \______________/
 协议       主机          路径     查询串 query（? 后面那一串）


逐块对应：

| 部分 | 例子 | 类比 |
|---|---|---|
| **scheme（协议）** | `https` | 用什么“语言”说话；`https` 是加密的 |
| **host（主机）** | `httpbin.org` | 哪栋楼（服务器域名） |
| **path（路径）** | `/get` | 楼里具体哪个房间 |
| **query（查询串）** | `?q=python&page=2` | 给前台/服务员的**备注** |

下面这段代码**完全不需要联网**，用 Python 把任意网址拆开给你看：

In [2]:
url = 'https://httpbin.org/get?q=python&page=2&sort=price'   # 随便写一个网址
parts = urlparse(url)            # urlparse 是标准库函数，把网址拆成几部分
print('scheme（协议）：', parts.scheme)
print('host（主机）：  ', parts.netloc)
print('path（路径）：  ', parts.path)
print('query（查询串）：', parts.query)

scheme（协议）： https
host（主机）：   httpbin.org
path（路径）：   /get
query（查询串）： q=python&page=2&sort=price


## 2.2 查询参数是什么：给服务器的“点菜备注”

**类比**：你去奶茶店点单“一杯奶茶，**备注：少冰、三分糖**”。奶茶是固定的商品（path），备注是你附加的个性化要求（query）。

**是什么**：`?` 后面一串 `键=值` 的组合，多个条件用 `&` 连接。例如 `?q=python&page=2` 的意思是：

- `q=python`：我要搜索关键词 python；
- `page=2`：我要看第 2 页。

**为什么需要它**：同一个页面 `/search`，靠不同的查询参数就能返回不同结果（搜不同词、翻不同页），不用为每页单独写一个网址。爬虫翻页、搜索，本质都是在改 query。

## 2.3 为什么不能手写查询串：中文和空格会出错

你可能想直接写：`'https://httpbin.org/get?q=编程 入门&page=2'`。

问题来了：网址里**不能直接放中文和空格**，必须转义成 `%xx` 这种形式（叫 URL 编码）。手写既麻烦又容易错。

**正确做法**：用 Python 的 `urlencode` 把一个字典自动拼好。这一步**完全离线**，断网也有输出：

In [3]:
params = {'q': '编程 入门', 'page': 2}   # 用字典表示备注，键值对清清楚楚
query_string = urlencode(params)          # 自动做 URL 编码，空格、中文都它来处理
print('拼好的查询串：', query_string)

# 再和基础网址拼成完整 URL（注意中间的 ?）
full_url = 'https://httpbin.org/get' + '?' + query_string
print('完整网址：', full_url)

拼好的查询串： q=%E7%BC%96%E7%A8%8B+%E5%85%A5%E9%97%A8&page=2
完整网址： https://httpbin.org/get?q=%E7%BC%96%E7%A8%8B+%E5%85%A5%E9%97%A8&page=2


## 2.4 requests 更省事：`params=` 参数自动拼

实际写爬虫时，你**不用自己拼 `?`**。只要把字典传给 `requests.get(..., params=...)`，它会自动调用 `urlencode` 拼好网址。

`httpbin.org/get` 是一个专门给程序员测试的网站，它会把“你发来的参数”**原样回显**给你，非常适合验证。联网部分用 `try/except` 包好，断网也不会报错：

In [4]:
try:
    # params 里写备注，网址里只写基础路径即可
    resp = requests.get(
        'https://httpbin.org/get',
        params={'q': 'python', 'page': 2},
        timeout=10                      # 最多等 10 秒，下节课展开讲
    )
    data = resp.json()                  # httpbin 返回的是 JSON 格式的回显
    print('状态码：', resp.status_code)
    print('服务器收到的参数 args：', data['args'])
    print('requests 实际请求的完整网址：', resp.url)
except Exception as e:
    print('这次没联网，不影响学习：', type(e).__name__, e)

状态码： 200
服务器收到的参数 args： {'page': '2', 'q': 'python'}
requests 实际请求的完整网址： https://httpbin.org/get?q=python&page=2


## 2.5 小结（记住这三句）

1. 网址 = **协议 scheme + 主机 host + 路径 path + 查询串 query**；`?` 后面就是查询参数。
2. 查询参数是给服务器的“备注”，用字典表示最清楚；**不要手写中文/空格，交给 `urlencode` 或 `params=`**。
3. `requests.get(url, params={...})` 自动拼好网址，`resp.url` 能看到它最终请求了哪个完整地址。

## 2.6 随堂小练习（自己动手，空格里写）

1. 用 `urlencode` 把字典 `{'keyword': '数据结构', 'page': 3, 'sort': 'asc'}` 拼成查询串，打印出来。
2. 用 `requests.get(..., params=...)` 请求 `https://httpbin.org/get`，带上参数 `{'name': '你的名字', 'age': 18}`，打印回显里的 `args`。

In [5]:
# TODO：自己写代码完成上面两题（这是随堂小练，没有单独答案文件）


## 来源与延伸

- requests 官方文档（传递参数）：https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls
- Python 标准库 urllib.parse：https://docs.python.org/3/library/urllib.parse.html
- MDN HTTP 概述：https://developer.mozilla.org/zh-CN/docs/Web/HTTP
- httpbin（回显请求的测试站）：https://httpbin.org
- 下一课：请求时除了网址，还能附一张“身份便签”——请求头。